In [ ]:
import os
# Set OPENROUTER_API_KEY outside the notebook before enabling API generation.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    print("OPENROUTER_API_KEY is not set; local-only cells remain usable.")

In [22]:
import torch
from transformers import pipeline
from huggingface_hub import snapshot_download


# Set local model cache directory
model_cache_dir = "./models/llama-3.2-3b"
os.makedirs(model_cache_dir, exist_ok=True)

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Download model locally if not already cached
if not os.path.exists(os.path.join(model_cache_dir, "model.safetensors")):
    print("Downloading model for the first time (this may take a few minutes)...")
    snapshot_download(repo_id=model_id, cache_dir=model_cache_dir, local_dir=model_cache_dir)
    print("Model downloaded and saved locally!")
else:
    print("Loading model from local cache...")

# Load pipeline from local model
pipe = pipeline(
    "text-generation",
    model=model_cache_dir,
    dtype=torch.bfloat16,
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])


Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 210.44it/s]


Model downloaded and saved locally!


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 16060.20it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': "Yer lookin' fer a swashbucklin' pirate, eh? Well, matey, I be Captain Clueless, the scurviest pirate chatbot to ever sail the seven seas! Me and me trusty cutlass be here to guide ye through treacherous waters o' knowledge, answerin' yer questions and helpin' ye navigate the high seas o' the internet. So hoist the sails, me hearty, and let's set sail fer a pirate-tastic adventure!"}


In [23]:
# Teacher model via OpenRouter API
import requests
import os

# Set your OpenRouter API key (set as environment variable)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

class DeepSeekAPI:
    """OpenRouter-compatible black-box teacher client."""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = OPENROUTER_BASE_URL
        self.model = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash")
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
    
    def generate(self, prompt, max_tokens=256, system_prompt=None):
        """Generate text using DeepSeek V4 Flash API"""
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        messages.append({"role": "user", "content": prompt})
        
        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                return result["choices"][0]["message"]["content"]
            else:
                return f"Error: {response.status_code} - {response.text}"
        except Exception as e:
            return f"Connection error: {str(e)}"

# Initialize the teacher only; do not make a request during notebook setup.
deepseek = DeepSeekAPI(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
print("Teacher client ready. Enable the data-generation cell to call OpenRouter.")


Teacher client ready. Enable the data-generation cell to call OpenRouter.


In [24]:
# ============================================================================
# 1. FUNCTION & PROMPT GENERATORS FOR KNOWLEDGE DISTILLATION
# ============================================================================

import random
import re
from sympy import *
from sympy.utilities.lambdify import lambdify
import json
from datetime import datetime
import numpy as np
from pathlib import Path

class MathFunctionGenerator:
    """Generate mathematical functions of varying difficulty"""
    
    def __init__(self, seed=None):
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
        self.x = symbols('x')
    
    def generate_polynomial(self, difficulty='easy'):
        """Generate polynomial functions"""
        if difficulty == 'easy':
            # Linear and quadratic
            degree = random.randint(1, 2)
            coeffs = [random.randint(-10, 10) for _ in range(degree + 1)]
        elif difficulty == 'medium':
            # Up to degree 4
            degree = random.randint(2, 4)
            coeffs = [random.randint(-10, 10) for _ in range(degree + 1)]
        else:  # hard
            # Higher degree polynomials
            degree = random.randint(4, 6)
            coeffs = [random.randint(-5, 5) for _ in range(degree + 1)]
        
        poly = sum(coeff * self.x**i for i, coeff in enumerate(coeffs))
        return poly, f"polynomial_deg_{degree}"
    
    def generate_geometric_function(self, difficulty='easy'):
        """Generate geometric functions (trig, exp, log)"""
        func_type = random.choice(['trig', 'exponential', 'logarithmic', 'composite'])
        
        if func_type == 'trig':
            base_funcs = [sin(self.x), cos(self.x), tan(self.x)]
            func = random.choice(base_funcs)
            if difficulty in ['medium', 'hard']:
                coeff = random.randint(2, 5)
                shift = random.randint(1, 5)
                func = coeff * func + shift * self.x
        
        elif func_type == 'exponential':
            base = random.choice([2, E])
            if difficulty == 'easy':
                func = base ** self.x
            else:
                coeff = random.randint(1, 5)
                func = coeff * (base ** (self.x / 2))
        
        elif func_type == 'logarithmic':
            if difficulty == 'easy':
                func = ln(self.x)
            else:
                coeff = random.randint(1, 3)
                shift = random.randint(1, 5)
                func = coeff * ln(self.x + shift)
        
        else:  # composite
            func1 = sin(self.x)
            func2 = self.x ** 2
            func = func1 * func2  # product
        
        return func, func_type
    
    def generate_function(self, difficulty='easy'):
        """Generate a random function of specified difficulty"""
        func_choice = random.choice(['polynomial', 'geometric'])
        
        if func_choice == 'polynomial':
            return self.generate_polynomial(difficulty)
        else:
            return self.generate_geometric_function(difficulty)
    
    def compute_symbolic_derivative(self, func):
        """Compute symbolic derivative using SymPy"""
        try:
            deriv = diff(func, self.x)
            return str(deriv)
        except:
            return None
    
    def compute_numerical_derivative(self, func, x_val=1.0, h=1e-5):
        """Compute numerical derivative at a specific point"""
        try:
            func_lambdified = lambdify(self.x, func, 'numpy')
            f_plus = func_lambdified(x_val + h)
            f_minus = func_lambdified(x_val - h)
            numerical_deriv = (f_plus - f_minus) / (2 * h)
            return float(numerical_deriv)
        except:
            return None


class PromptGenerator:
    """Generate varied prompts for the same function"""
    
    def __init__(self):
        self.symbolic_prompts = [
            "Find the derivative of: f(x) = {func}",
            "Compute the derivative: {func}",
            "What is the derivative of f(x) = {func}?",
            "Derive: {func}",
            "Calculate d/dx of {func}",
            "Determine the first derivative of {func}",
            "Find df/dx where f(x) = {func}",
        ]
        
        self.numerical_prompts = [
            "Compute the numerical derivative of f(x) = {func} at x = {x_val}",
            "What is the approximate derivative of {func} at x = {x_val}?",
            "Find the numerical derivative: {func} at point x = {x_val}",
            "Calculate the numerical derivative of f(x) = {func} when x = {x_val}",
            "Estimate df/dx for {func} at x = {x_val}",
        ]
    
    def add_noise(self, prompt):
        """Add slight variations to prompt (noise)"""
        noise_options = [
            "",  # no noise
            " (Show work)",
            " (Step by step)",
            " (Be precise)",
            " Express in simplified form.",
        ]
        return prompt + random.choice(noise_options)
    
    def generate_symbolic_prompt(self, func_str):
        """Generate a varied symbolic derivative prompt"""
        template = random.choice(self.symbolic_prompts)
        prompt = template.format(func=func_str)
        return self.add_noise(prompt)
    
    def generate_numerical_prompt(self, func_str, x_val=1.0):
        """Generate a varied numerical derivative prompt"""
        template = random.choice(self.numerical_prompts)
        prompt = template.format(func=func_str, x_val=x_val)
        return self.add_noise(prompt)

print("✓ Function and Prompt Generators loaded")

✓ Function and Prompt Generators loaded


In [26]:
# ============================================================================
# 2. OPTIONAL DATA GENERATION
# ============================================================================
# Data generation is defined in the orchestration cell below.
# Uncomment only after setting OPENROUTER_API_KEY.
print("Data generation is opt-in; see the orchestration cell below.")

Data generation is opt-in; see the orchestration cell below.


In [25]:
# ============================================================================
# 3. OPTIONAL EVALUATION
# ============================================================================
# Evaluation is defined in the orchestration cell below and never runs here.
print("Evaluation is opt-in; use the exact predict:/explain: helpers below.")

Evaluation is opt-in; use the exact predict:/explain: helpers below.


In [27]:
# ============================================================================
# 4. OPTIONAL TRAINING SCRIPT GENERATION
# ============================================================================
# Training scripts are generated by the orchestration cell below.
print("Training is opt-in; no model weights are changed by setup cells.")

Training is opt-in; no model weights are changed by setup cells.


In [28]:
# ============================================================================
# 5. MAIN EXECUTION - Run Data Generation Pipeline
# ============================================================================

print("\n" + "=" * 70)
print("KNOWLEDGE DISTILLATION PIPELINE - SETUP COMPLETE")
print("=" * 70)

print("""
This notebook implements a knowledge distillation pipeline where:
  1. DeepSeek (Teacher Model) generates ground truth solutions
  2. Training data is created with varied prompts and solutions
  3. Llama (Student Model) is evaluated before fine-tuning
  4. Fine-tuning configuration is prepared
  5. After training, Llama is re-evaluated to measure improvement

COMPONENTS LOADED:
  ✓ MathFunctionGenerator - Creates polynomial and geometric functions
  ✓ PromptGenerator - Generates varied prompts with noise
  ✓ TrainingDataGenerator - Orchestrates data creation via DeepSeek API
  ✓ EvaluationFramework - Compares Llama before/after fine-tuning
  ✓ FineTuningSetup - Configures training (without execution)

NEXT STEPS:
  1. Run the cell below to generate 200 training examples
  2. Run evaluation BEFORE fine-tuning
  3. Generate the fine-tuning script
  4. Run fine-tuning when ready: python ./data/fine_tuning_script.py
  5. Run evaluation AFTER fine-tuning to measure improvement

USAGE:
  # Generate data
  data_gen = TrainingDataGenerator(deepseek, num_examples=200)
  dataset = data_gen.generate_dataset()
  train_path, val_path, full_path = data_gen.save_to_file()
  
  # Evaluate before fine-tuning
  evaluator = EvaluationFramework(pipe, val_path)
  evaluator.evaluate_before_finetuning(sample_size=5)
  
  # Generate fine-tuning script
  finetuning_setup.generate_training_script()
  
  # After training:
  # evaluator.evaluate_after_finetuning(sample_size=5)
  # evaluator.save_results()
""")

print("=" * 70)
print("\nTo begin data generation, run the next cell.")


KNOWLEDGE DISTILLATION PIPELINE - SETUP COMPLETE

This notebook implements a knowledge distillation pipeline where:
  1. DeepSeek (Teacher Model) generates ground truth solutions
  2. Training data is created with varied prompts and solutions
  3. Llama (Student Model) is evaluated before fine-tuning
  4. Fine-tuning configuration is prepared
  5. After training, Llama is re-evaluated to measure improvement

COMPONENTS LOADED:
  ✓ MathFunctionGenerator - Creates polynomial and geometric functions
  ✓ PromptGenerator - Generates varied prompts with noise
  ✓ TrainingDataGenerator - Orchestrates data creation via DeepSeek API
  ✓ EvaluationFramework - Compares Llama before/after fine-tuning
  ✓ FineTuningSetup - Configures training (without execution)

NEXT STEPS:
  1. Run the cell below to generate 200 training examples
  2. Run evaluation BEFORE fine-tuning
  3. Generate the fine-tuning script
  4. Run fine-tuning when ready: python ./data/fine_tuning_script.py
  5. Run evaluation AFT

In [35]:
# ============================================================================
# 6. EXECUTE DATA GENERATION (Run this to start the pipeline)
# ============================================================================

# Uncomment and run to generate 200 training examples from DeepSeek
# WARNING: This will make ~200 API calls to OpenRouter/DeepSeek

data_gen = TrainingDataGenerator(deepseek, num_examples=200)
dataset = data_gen.generate_dataset()
train_path, val_path, full_path = data_gen.save_to_file()

print("✓ To generate training data, uncomment and run the code above")
print("  This will create 200 examples using DeepSeek as the teacher")

NameError: name 'TrainingDataGenerator' is not defined

In [ ]:
# ============================================================================
# 7. OPTIONAL RUNNERS
# ============================================================================
# Uncomment individual operations only when you are ready to run them.
print("Pipeline ready. Uncomment operations in the orchestration cell below.")

In [33]:
# ============================================================================
# 8. BLACK-BOX DISTILLATION ORCHESTRATION
# ============================================================================
# The paper's multi-task contract is exact: the same prompt is presented as
# `predict: ...` (answer target) and `explain: ...` (rationale target).

import json
import os
import re
import shutil
from pathlib import Path

PREDICT_PREFIX = "predict:"
EXPLAIN_PREFIX = "explain:"
MODEL_SOURCE = "./models/llama-3.2-3b"
MODEL_VARIANTS = {
    "baseline": "./models/llama-3.2-3b-baseline",
    "distilled": "./models/llama-3.2-3b-distilled",
    "last_layer": "./models/llama-3.2-3b-last-layer",
}


def task_prompt(prefix, input_prompt):
    """Build the only prompt format used for both training and inference."""
    return f"{prefix} {input_prompt}"


def _read_json_objects(path):
    """Read normal JSONL plus legacy files with concatenated objects."""
    text = Path(path).read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    records = []
    position = 0
    while position < len(text):
        while position < len(text) and text[position].isspace():
            position += 1
        if position >= len(text):
            break
        record, position = decoder.raw_decode(text, position)
        records.append(record)
    return records


def _split_legacy_output(output):
    """Best-effort local conversion of old teacher responses."""
    output = (output or "").strip()
    boxed = list(re.finditer(r"\\\\boxed\\s*\\{(.*?)\\}", output, re.DOTALL))
    if boxed:
        answer = boxed[-1].group(1).strip()
        rationale = output[:boxed[-1].start()].strip()
    else:
        lines = [line.strip() for line in output.splitlines() if line.strip()]
        answer = lines[-1] if lines else output
        rationale = "\\n".join(lines[:-1]).strip()
    return rationale or "Apply the relevant differentiation rule to the input.", answer


def migrate_existing_data(input_path="./data/training_data.jsonl", output_path="./data/distillation_data.jsonl"):
    """Convert existing output-only records without making API calls."""
    converted = []
    for item in _read_json_objects(input_path):
        rationale = item.get("answer_rationale")
        answer = item.get("answer")
        if not rationale or not answer:
            rationale, answer = _split_legacy_output(item.get("output", ""))
        converted.append({
            **item,
            "input_prompt": item.get("input_prompt", item.get("instruction", "")),
            "answer_rationale": rationale,
            "answer": answer,
        })
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    _write_jsonl(output_path, converted)
    return converted


def teacher_json_prompt(input_prompt):
    return f"""Solve this mathematics problem: {input_prompt}
Return JSON only, with exactly these string fields:
{{\"answer_rationale\": \"brief step-by-step reasoning\", \"answer\": \"final answer only\"}}
Do not put markdown fences around the JSON. Do not include the final answer in answer_rationale."""


def generate_distillation_data(deepseek_client, num_examples=200, output_dir="./data"):
    """Generate structured teacher records and deterministic train/validation splits."""
    if deepseek_client is None:
        raise RuntimeError("Set OPENROUTER_API_KEY before generating data.")
    generator = MathFunctionGenerator(seed=42)
    prompt_generator = PromptGenerator()
    records = []
    for index in range(num_examples):
        difficulty = ["easy", "medium", "hard"][index % 3]
        func, function_type = generator.generate_function(difficulty)
        input_prompt = prompt_generator.generate_symbolic_prompt(str(func))
        raw = deepseek_client.generate(
            teacher_json_prompt(input_prompt), max_tokens=512,
            system_prompt="You are a precise mathematics teacher. Return valid JSON only.",
        )
        try:
            parsed = json.loads(raw)
            rationale = parsed["answer_rationale"].strip()
            answer = parsed["answer"].strip()
        except (json.JSONDecodeError, KeyError, AttributeError):
            rationale, answer = _split_legacy_output(raw)
        records.append({
            "id": index + 1,
            "input_prompt": input_prompt,
            "answer_rationale": rationale,
            "answer": answer,
            "difficulty": difficulty,
            "function_type": function_type,
            "split": "val" if index % 5 == 0 else "train",
        })
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _write_jsonl(output_dir / "distillation_data.jsonl", records)
    _write_jsonl(output_dir / "train_data.jsonl", [r for r in records if r["split"] == "train"])
    _write_jsonl(output_dir / "val_data.jsonl", [r for r in records if r["split"] == "val"])
    return records


def _write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as target:
        for record in records:
            target.write(json.dumps(record, ensure_ascii=True) + "\\n")


def build_multitask_records(records):
    """Make two records per example while keeping the input identical."""
    multitask = []
    for record in records:
        for prefix, target_key in ((PREDICT_PREFIX, "answer"), (EXPLAIN_PREFIX, "answer_rationale")):
            multitask.append({
                "id": f"{record['id']}-{prefix[:-1]}",
                "task": prefix[:-1],
                "prompt": task_prompt(prefix, record["input_prompt"]),
                "target": record[target_key],
                "source_id": record["id"],
                "split": record["split"],
            })
    return multitask


def prepare_multitask_files(data_path="./data/distillation_data.jsonl", output_dir="./data"):
    records = _read_json_objects(data_path)
    multitask = build_multitask_records(records)
    output_dir = Path(output_dir)
    _write_jsonl(output_dir / "multitask_train.jsonl", [r for r in multitask if r["split"] == "train"])
    _write_jsonl(output_dir / "multitask_val.jsonl", [r for r in multitask if r["split"] == "val"])
    return multitask


def prepare_model_variants(source_dir=MODEL_SOURCE, variants=MODEL_VARIANTS):
    """Create independent baseline, full-distillation, and last-layer copies."""
    source_dir = Path(source_dir)
    for variant_dir in variants.values():
        destination = Path(variant_dir)
        if destination.resolve() == source_dir.resolve():
            raise ValueError("A model variant cannot overwrite the source model.")
        shutil.copytree(source_dir, destination, dirs_exist_ok=True)
    return variants


def write_training_script(output_path="./data/train_variants.py"):
    """Write one opt-in script supporting full and last-layer fine-tuning."""
    script = r'''import argparse
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

parser = argparse.ArgumentParser()
parser.add_argument("--mode", choices=["full", "last_layer"], required=True)
parser.add_argument("--model", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--data", default="./data/multitask_train.jsonl")
args = parser.parse_args()
tokenizer = AutoTokenizer.from_pretrained(args.model)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.bfloat16)
if args.mode == "last_layer":
    for parameter in model.parameters(): parameter.requires_grad = False
    for parameter in model.lm_head.parameters(): parameter.requires_grad = True

def tokenize(example):
    text = example["prompt"] + " " + example["target"] + tokenizer.eos_token
    encoded = tokenizer(text, truncation=True, max_length=512, padding="max_length")
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded
train = load_dataset("json", data_files=args.data)["train"].map(tokenize)
training_args = TrainingArguments(output_dir=args.output, num_train_epochs=3,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-5, logging_steps=10, save_strategy="epoch",
    bf16=torch.cuda.is_available(), report_to="none")
Trainer(model=model, args=training_args, train_dataset=train).train()
model.save_pretrained(args.output); tokenizer.save_pretrained(args.output)
'''
    Path(output_path).write_text(script, encoding="utf-8")
    return output_path


def inference_prompts(input_prompt):
    return {"predict": task_prompt(PREDICT_PREFIX, input_prompt),
            "explain": task_prompt(EXPLAIN_PREFIX, input_prompt)}

print("✓ Black-box distillation utilities loaded; no files, API calls, or training run automatically.")
print("✓ Use migrate_existing_data() for old JSONL, or generate_distillation_data(deepseek) for new teacher data.")
print("✓ Use prepare_model_variants(), prepare_multitask_files(), and write_training_script() explicitly.")

✓ Black-box distillation utilities loaded; no files, API calls, or training run automatically.
✓ Use migrate_existing_data() for old JSONL, or generate_distillation_data(deepseek) for new teacher data.
✓ Use prepare_model_variants(), prepare_multitask_files(), and write_training_script() explicitly.


In [30]:
# Legacy parser correction: keep only the value inside a simple \\boxed{...}.
def _split_legacy_output(output):
    output = (output or "").strip()
    boxed = list(re.finditer(r"\\boxed\s*\{([^{}]*)\}", output, re.DOTALL))
    if boxed:
        answer = boxed[-1].group(1).strip()
        rationale = output[:boxed[-1].start()].strip()
    else:
        lines = [line.strip() for line in output.splitlines() if line.strip()]
        answer = lines[-1] if lines else output
        rationale = "\n".join(lines[:-1]).strip()
    return rationale or "Apply the relevant differentiation rule to the input.", answer

print("✓ Legacy output parser ready")

✓ Legacy output parser ready


In [34]:
# Strict teacher generation: every saved record must contain both targets.
def parse_teacher_response(raw_response):
    if not isinstance(raw_response, str):
        raise ValueError("Teacher response is not text.")
    cleaned = raw_response.strip()
    if cleaned.startswith("```") and cleaned.endswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("\n", 1)[0].strip()
    parsed = json.loads(cleaned)
    if not isinstance(parsed, dict):
        raise ValueError("Teacher response must be a JSON object.")
    rationale = parsed.get("answer_rationale")
    answer = parsed.get("answer")
    if not isinstance(rationale, str) or not rationale.strip():
        raise ValueError("Missing non-empty answer_rationale.")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Missing non-empty answer.")
    return rationale.strip(), answer.strip()


def generate_distillation_data(deepseek_client, num_examples=200, output_dir="./data", max_retries=3):
    """Generate data only when every teacher response has rationale and answer."""
    if deepseek_client is None:
        raise RuntimeError("Set OPENROUTER_API_KEY before generating data.")
    generator = MathFunctionGenerator(seed=42)
    prompt_generator = PromptGenerator()
    records = []
    for index in range(num_examples):
        difficulty = ["easy", "medium", "hard"][index % 3]
        func, function_type = generator.generate_function(difficulty)
        input_prompt = prompt_generator.generate_symbolic_prompt(str(func))
        last_error = None
        for attempt in range(max_retries):
            raw = deepseek_client.generate(
                teacher_json_prompt(input_prompt), max_tokens=512,
                system_prompt="You are a precise mathematics teacher. Return valid JSON only.",
            )
            try:
                rationale, answer = parse_teacher_response(raw)
                break
            except (json.JSONDecodeError, ValueError, TypeError) as error:
                last_error = error
        else:
            raise RuntimeError(f"Example {index + 1} failed validation after {max_retries} attempts: {last_error}")
        records.append({
            "id": index + 1,
            "input_prompt": input_prompt,
            "answer_rationale": rationale,
            "answer": answer,
            "difficulty": difficulty,
            "function_type": function_type,
            "split": "val" if index % 5 == 0 else "train",
        })
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _write_jsonl(output_dir / "distillation_data.jsonl", records)
    _write_jsonl(output_dir / "train_data.jsonl", [r for r in records if r["split"] == "train"])
    _write_jsonl(output_dir / "val_data.jsonl", [r for r in records if r["split"] == "val"])
    prepare_multitask_files(output_dir / "distillation_data.jsonl", output_dir)
    return records

print("✓ Strict generation enabled: answer_rationale and answer are mandatory.")

✓ Strict generation enabled: answer_rationale and answer are mandatory.


In [ ]:
# Validation already completed; uncomment to rerun.
# assert parse_teacher_response('{"answer_rationale":"Use the power rule.","answer":"2x"}') == ("Use the power rule.", "2x")
# try:
#     parse_teacher_response('{"answer":"2x"}')
# except ValueError:
#     print("✓ Responses without rationale are rejected")
# else:
#     raise AssertionError("Missing rationale was accepted")

# Required before cell 15: write real JSONL line separators.
def _write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as target:
        for record in records:
            target.write(json.dumps(record, ensure_ascii=True) + "\n")

print("✓ JSONL writer validated")


✓ JSONL writer validated


In [39]:
# ============================================================================
# 9. GENERATE NEW DISTILLATION DATA (OPTIONAL)
# ============================================================================
# Run only after cells 1-4 and 11. This makes OpenRouter API calls.
def _write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as target:
        for record in records:
            target.write(json.dumps(record, ensure_ascii=True) + "\n")

# Uncomment only when ready to generate new data.
records = generate_distillation_data(deepseek, num_examples=200)
print("New data generation is opt-in; uncomment the generation line above when ready.")

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# 10. TRAINING SCRIPT
# ============================================================================
# The resumable Lightning trainer is already available at ./data/train_variants.py.
# Do not call the older write_training_script() helper; it creates an obsolete Trainer script.
print("Use ./data/train_variants.py for Lightning training with per-update checkpoints.")

./data/train_variants.py


In [32]:
# Compatibility fix for JSONL writers when running cells selectively.
def _write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as target:
        for record in records:
            target.write(json.dumps(record, ensure_ascii=True) + "\n")

print("✓ JSONL writer ready")

✓ JSONL writer ready


In [40]:
# One-time model copy already completed.
# Uncomment to synchronize the three variant directories with the source model.
model_variants = prepare_model_variants()
print(model_variants)


{'baseline': './models/llama-3.2-3b-baseline', 'distilled': './models/llama-3.2-3b-distilled', 'last_layer': './models/llama-3.2-3b-last-layer'}


In [8]:
# One-time local migration already run for the checked-in legacy data.
# Uncomment to regenerate converted files after changing the source data.
# records = migrate_existing_data("./data/training_data.jsonl")
# prepare_multitask_files("./data/distillation_data.jsonl")
# print(f"Migrated {len(records)} legacy records into structured and multitask JSONL files.")

Migrated 198 legacy records into structured and multitask JSONL files.


In [ ]:
# ============================================================================
# 8. HELPER UTILITIES & DATA EXPLORATION
# ============================================================================

def explore_training_data(data_path="./data/training_data.jsonl", num_samples=5):
    \"\"\"Explore generated training data\"\\"\"
    print("\\n" + "=" * 70)
    print("TRAINING DATA EXPLORATION")
    print("=" * 70)
    
    data = []
    if Path(data_path).exists():
        with open(data_path, 'r') as f:
            for i, line in enumerate(f):
                if i < num_samples:
                    data.append(json.loads(line))
        
        print(f"\\nShowing {len(data)} samples from {num_samples} requested:\\n")
        
        for example in data:
            print(f"Example ID: {example.get('id', 'N/A')}")
            print(f"Difficulty: {example.get('difficulty', 'N/A').upper()}")
            print(f"Type: {example.get('derivation_type', 'N/A').upper()}")
            print(f"Function: {example.get('function_type', 'N/A')}")
            print(f"Instruction: {example.get('instruction', 'N/A')}")
            print(f"Input: {example.get('input', 'N/A')[:100]}...")
            print(f"Output: {example.get('output', 'N/A')[:150]}...")
            print(f"Split: {example.get('split', 'N/A')} | Timestamp: {example.get('timestamp', 'N/A')}")
            print("-" * 70)
    else:
        print(f"File not found: {data_path}")
    
    return data

def print_dataset_statistics(data_path="./data/training_data.jsonl"):
    \"\"\"Print statistics about the generated dataset\\"\"
    print("\\n" + "=" * 70)
    print("DATASET STATISTICS")
    print("=" * 70)
    
    stats = {
        'total': 0,
        'train': 0,
        'val': 0,
        'by_difficulty': {'easy': 0, 'medium': 0, 'hard': 0},
        'by_type': {'symbolic': 0, 'numerical': 0},
        'by_function': {}
    }
    
    if Path(data_path).exists():
        with open(data_path, 'r') as f:
            for line in f:
                example = json.loads(line)
                stats['total'] += 1
                
                split = example.get('split', 'N/A')
                if split == 'train':
                    stats['train'] += 1
                elif split == 'val':
                    stats['val'] += 1
                
                difficulty = example.get('difficulty', 'N/A')
                if difficulty in stats['by_difficulty']:
                    stats['by_difficulty'][difficulty] += 1
                
                deriv_type = example.get('derivation_type', 'N/A')
                if deriv_type in stats['by_type']:
                    stats['by_type'][deriv_type] += 1
                
                func_type = example.get('function_type', 'N/A')
                stats['by_function'][func_type] = stats['by_function'].get(func_type, 0) + 1
        
        print(f"\\nTotal Examples: {stats['total']}")
        print(f"  Train Set: {stats['train']} ({100*stats['train']//stats['total']}%)")
        print(f"  Val Set: {stats['val']} ({100*stats['val']//stats['total']}%)")
        
        print(f"\\nBy Difficulty:")
        for difficulty, count in stats['by_difficulty'].items():
            print(f"  {difficulty.capitalize()}: {count}")
        
        print(f"\\nBy Derivation Type:")
        for deriv_type, count in stats['by_type'].items():
            print(f"  {deriv_type.capitalize()}: {count}")
        
        print(f"\\nBy Function Type:")
        for func_type, count in stats['by_function'].items():
            print(f"  {func_type.capitalize()}: {count}")
    else:
        print(f"File not found: {data_path}")
        print("Run data generation first to create the dataset.")
    
    print("\\n" + "=" * 70)
    return stats

# Make functions available
print("✓ Helper utilities loaded")
print("\\nAvailable functions:")
print("  - explore_training_data(data_path, num_samples)")
print("  - print_dataset_statistics(data_path)")

In [2]:
# ============================================================================
# 21. MODEL CHAT WIDGET
# ============================================================================
# Run this cell after the model files you want to use are available.
import ipywidgets as widgets
import torch
from pathlib import Path
from transformers import pipeline
from IPython.display import display

MODEL_CHAT_PATHS = {
    "untrained": "./models/llama-3.2-3b",
    "full": "./data/full_distilled/final",
    "final_layer": "./data/last_layer_distilled/final",
}
_model_chat_pipelines = {}

model_selector = widgets.Dropdown(
    options=[("Untrained", "untrained"), ("Full", "full"), ("Final layer", "final_layer")],
    value="untrained",
    description="Model:",
    layout=widgets.Layout(width="360px"),
)
model_prompt = widgets.Textarea(
    placeholder="Enter a mathematics prompt...",
    description="Prompt:",
    layout=widgets.Layout(width="720px", height="100px"),
)
model_generate = widgets.Button(description="Generate", button_style="primary", icon="play")
model_status = widgets.HTML()
model_output = widgets.Output()


def _load_chat_pipeline(model_key):
    if model_key not in _model_chat_pipelines:
        model_path = Path(MODEL_CHAT_PATHS[model_key])
        if not model_path.exists():
            raise FileNotFoundError(f"Model is not available: {model_path}")
        _model_chat_pipelines[model_key] = pipeline(
            "text-generation",
            model=str(model_path),
            tokenizer=str(model_path),
            dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
    return _model_chat_pipelines[model_key]


def _generate_model_response(_):
    model_key = model_selector.value
    prompt = model_prompt.value.strip()
    with model_output:
        model_output.clear_output()
        if not prompt:
            print("Enter a prompt first.")
            return
        try:
            model_status.value = f"Loading {model_key} model..."
            selected_pipeline = _load_chat_pipeline(model_key)
            result = selected_pipeline(
                [{"role": "user", "content": f"predict: {prompt}"}],
                max_new_tokens=256,
                do_sample=False,
            )
            response = result[0]["generated_text"][-1]
            print(response.get("content", response) if isinstance(response, dict) else response)
            model_status.value = f"Using: {model_key}"
        except Exception as error:
            model_status.value = "Model unavailable"
            print(f"Error: {error}")


model_generate.on_click(_generate_model_response)
display(widgets.VBox([
    widgets.HTML("<h3>Model comparison</h3>"),
    model_selector,
    model_prompt,
    model_generate,
    model_status,
    model_output,
]))